# Spark Declarative Pipelines — build a medallion in SQL

**⏱ ~4 min  ·  🎯 Show that you *declare* tables in SQL and SDP builds, orders, and validates the pipeline  ·  🧰 Setup: none — `Shift`+`Enter` through each cell.**

> **Before you start:** run the **warm-up** cell below once (it starts Spark, ~10s). Talk through this intro while it loads.

**What the audience will see:** a bronze → silver → gold pipeline over real food-delivery orders, written almost entirely in SQL, landing Delta tables in Unity Catalog — and SDP catching a bug *before* it runs.

In [ ]:
# Warm-up: start Spark (run once, ~10s)
from pyspark.sql import SparkSession
from IPython.display import Code
import os
spark = SparkSession.builder.getOrCreate()
NS = os.environ.get('DEMO_NS','medallion_demo').rstrip('_') or 'medallion_demo'
print('ready — your schema is managed_demo.' + NS)

## 1. The pipeline is just a few files
An SDP pipeline = a spec + transformation files. One Python file ingests raw data; the rest is SQL.

> 💬 **Say:** "A pipeline here isn't a big script — it's a folder of small SQL files, one per table."
> 👀 **Point at:** the file list below.

In [ ]:
!ls -1 /home/jovyan/demos/sdp-medallion/transformations

Here is the **silver** layer — parse the JSON order body, add time features, join the city dimension. It's one `SELECT`. SDP reads the `FROM`/`JOIN` and wires the dependency itself.

> 💬 **Say:** "Notice there's no orchestration and no write code. I just say *what* this table is."

In [ ]:
Code(filename='/home/jovyan/demos/sdp-medallion/transformations/silver_orders_enriched.sql')

## 2. The payoff — SDP validates the whole graph *before* running
Here's a pipeline with a deliberate typo (`orderz` instead of `orders`). A **dry-run** checks the dependency graph without moving any data.

> 💬 **Say:** "With normal Spark, this typo crashes halfway through a long job. Watch SDP catch it instantly."
> 👀 **Point at:** the `orderz` line in the file, then in the error.

In [ ]:
Code(filename='/home/jovyan/demos/sdp-medallion/broken/transformations/gold_revenue.sql')

In [ ]:
!spark-pipelines dry-run --spec /home/jovyan/demos/sdp-medallion/broken/spark-pipeline.yml 2>&1 | grep -iE 'NOT_FOUND|Failed to resolve|orderz' || true

> 💬 **Say:** "Caught — exact table, exact line, before a single byte moved. That's the value of declaring your dependencies."

## 3. Run the real medallion
SDP runs the layers in dependency order and registers Delta tables in your schema.

> 💬 **Say:** "Now the real pipeline — bronze, silver, gold, in order, with no orchestration code from me."
> ⚠️ This takes ~1–1.5 min; narrate the layers as they complete. _(Re-running? Run the Reset cell at the bottom first.)_

In [ ]:
!spark-pipelines run --spec /home/jovyan/demos/sdp-medallion/spark-pipeline.yml 2>&1 | grep -E 'is COMPLETED|Run is COMPLETED'

## 4. The result — straight from Unity Catalog
> 👀 **Point at:** the chart — real revenue per brand, computed by the pipeline you just ran.

In [ ]:
bs = spark.table(f'managed_demo.{NS}.gold_brand_summary').orderBy('total_revenue', ascending=False).toPandas()
bs

In [ ]:
import matplotlib.pyplot as plt
top = bs.head(10)[::-1]
plt.figure(figsize=(8,4)); plt.barh(top['brand_name'], top['total_revenue'])
plt.xlabel('revenue'); plt.title(f'Revenue by brand — managed_demo.{NS}'); plt.tight_layout(); plt.show()

### Recap (say this to close)
- The transforms were plain SQL `SELECT`s — **SDP inferred the DAG, ordered execution, and handled every write.**
- `dry-run` caught a bad dependency **before any data moved**.
- The gold tables are in Unity Catalog under your schema — open the UC console to browse them.

---
#### Reset (only if you want to run the pipeline again)
Unity Catalog has no `TRUNCATE`, so drop the tables before a second run:

In [ ]:
!python3 /home/jovyan/demos/sdp-medallion/reset.py